In [ ]:
!pip install bertopic --quiet
!pip install llama-stack --quiet
!pip install llama-stack -U --quiet
!pip install -U bitsandbytes --quiet
!pip install bitsandbytes --quiet

In [ ]:
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from tqdm import tqdm
import numpy as np
import umap
import hdbscan
from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
df = pd.read_csv("/content/entrep+crypto_lit.csv")
df.head()

In [ ]:
abstracts = df['Abstract']
titles = df['Title']
year = df['Year']

In [ ]:
len(abstracts)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from torch import cuda

model_id = 'meta-llama/Llama-2-7b-chat-hf'
device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

print(device)

In [ ]:
from torch import bfloat16
import transformers

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)

In [ ]:
# Llama 2 Tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

# Llama 2 Model
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()

In [ ]:
generator = transformers.pipeline(
    model=model, tokenizer=tokenizer,
    task='text-generation',
    temperature=0.1,
    max_new_tokens=500,
    repetition_penalty=1.1
)

In [ ]:
prompt = "Could you explain to me how 4-bit quantization works as if I am 5?"
res = generator(prompt)
print(res[0]["generated_text"])

In [ ]:
system_prompt = """
<s>[INST] <<SYS>>
You are an expert academic researcher specializing in entrepreneurship and cryptocurrency research. Your task is to create concise, descriptive labels for research topics based on academic abstracts.
<</SYS>>
"""

example_prompt = """
I have a topic that contains the following academic abstracts:
- This study examines how blockchain-based initial coin offerings (ICOs) provide alternative fundraising mechanisms for technology startups, analyzing token sale structures and investor participation patterns across 250 ICO campaigns.
- We investigate the determinants of successful token offerings in the cryptocurrency market, focusing on whitepaper quality, team composition, and regulatory compliance as signals of venture legitimacy.
- Our research explores how entrepreneurs utilize tokenization to attract capital, examining the relationship between token economics design and fundraising outcomes in decentralized finance ventures.

The topic is described by the following keywords: 'ICO, token, funding, investor, capital, offering, blockchain, fundraising, venture'.

Based on the information about the topic above, please create a short label of this topic. Make sure to only return the label and nothing more.

[/INST] ICO Fundraising & Token Economics
"""

main_prompt = """
[INST]
I have a topic that contains the following academic abstracts:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information about the topic above, please create a short, descriptive label of this research topic. The label should:
- Be concise
- Capture the core research focus
- Use academic terminology appropriate for entrepreneurship and cryptocurrency research
- Be suitable for a literature review taxonomy

Make sure to only return the label and nothing more.
[/INST]
"""

In [ ]:
prompt = system_prompt + example_prompt + main_prompt

In [ ]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer("BAAI/bge-small-en")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

In [ ]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=20, min_samples=5, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [ ]:
# Pre-reduce embeddings for visualization purposes
reduced_embeddings = UMAP(n_neighbors=15, n_components=2, min_dist=0.1, metric='cosine', random_state=42).fit_transform(embeddings)

In [ ]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

# KeyBERT
keybert = KeyBERTInspired()

# MMR
mmr = MaximalMarginalRelevance(diversity=0.3)

# Text generation with Llama 2
llama2 = TextGeneration(generator, prompt=prompt, doc_length=50, tokenizer=tokenizer)

# All representation models
representation_model = {
    "KeyBERT": keybert,
    "Llama2": llama2,
    "MMR": mmr
}

In [ ]:
topic_model = BERTopic(

  # Sub-models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(abstracts, embeddings)

In [ ]:
topic_model.get_topic(1, full=True)["KeyBERT"]

In [ ]:
llama2_labels = [label[0][0].split("\n")[0] for label in topic_model.get_topics(full=True)["Llama2"].values()]
topic_model.set_topic_labels(llama2_labels)

In [ ]:
#topic_model.visualize_documents(titles, reduced_embeddings=reduced_embeddings, hide_annotations=True, hide_document_hover=False, custom_labels=True)

In [ ]:
topic_model.get_topic_info()

In [ ]:
#topic_info = topic_model.get_topic_info()
#topic_info.to_csv('bertopic_topics.csv', index=False)
#print('Topic information saved to bertopic_topics.csv')

In [ ]:
df['Topic'] = topics
df['Probability'] = probs

In [ ]:
# 1. Built-in temporal visualization
years = df['Year'].tolist()
topics_over_time = topic_model.topics_over_time(abstracts, years)
fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=14)
fig.write_html('all_topics_over_time.html')

In [ ]:
# 2. Simple period comparison
df['Period'] = df['Year'].apply(lambda y:
    "Early (2017-2019)" if y <= 2019 else
    "Mid (2020-2022)" if y <= 2022 else
    "Recent (2023+)")

In [ ]:
period_topic_table = df.groupby(['Period', 'Topic']).size().unstack(fill_value=0)
print("\n=== Papers per Topic per Period ===")
print(period_topic_table)

In [ ]:
topic_labels_dict = {i: topic_model.custom_labels_[i] for i in range(len(topic_model.custom_labels_))}
period_topic_table.columns = [topic_labels_dict.get(col, f"Topic {col}") for col in period_topic_table.columns]
print("\n=== Papers per Topic per Period (with labels) ===")
print(period_topic_table)

In [ ]:
period_topic_pct = df.groupby(['Period', 'Topic']).size().groupby(level=0).apply(lambda x: 100 * x / x.sum()).unstack(fill_value=0)
period_topic_pct.columns = [topic_labels_dict.get(col, f"Topic {col}") for col in period_topic_pct.columns]
print("\n=== Percentage of Papers per Period ===")
print(period_topic_pct.round(1))

In [ ]:
topic_summary = []
for topic in df['Topic'].unique():
    if topic == -1:  # Skip outliers
        continue

    topic_papers = df[df['Topic'] == topic]

    summary = {
        'Topic_Number': topic,
        'Topic_Label': topic_labels_dict[topic],
        'Total_Papers': len(topic_papers),
        'First_Year': topic_papers['Year'].min(),
        'Last_Year': topic_papers['Year'].max(),
        'Peak_Year': topic_papers.groupby('Year').size().idxmax(),
        'Peak_Count': topic_papers.groupby('Year').size().max(),
        'Early_Period_Papers': len(topic_papers[topic_papers['Period'] == 'Early (2017-2019)']),
        'Mid_Period_Papers': len(topic_papers[topic_papers['Period'] == 'Mid (2020-2021)']),
        'Recent_Period_Papers': len(topic_papers[topic_papers['Period'] == 'Recent (2022+)']),
        'Recent_Percentage': (len(topic_papers[topic_papers['Period'] == 'Recent (2022+)']) / len(topic_papers) * 100),
        'Average_Citations': topic_papers['Cited by'].mean() if 'Cited by' in topic_papers.columns else None
    }

    topic_summary.append(summary)

topic_summary_df = pd.DataFrame(topic_summary)
topic_summary_df = topic_summary_df.sort_values('Total_Papers', ascending=False)
topic_summary_df.to_csv('topic_summary_statistics.csv', index=False)
print("✓ Saved: topic_summary_statistics.csv")